# RAG 第 12 课：真实版 Reranking

上一课我们已经把真实数据、真实 embedding、真实本地 LLM 跑通了。
这一课继续往前走，把 pipeline 升级成更接近真实工程的两阶段结构：

1. 第一阶段：embedding 检索做粗召回
2. 第二阶段：LLM reranker 做精排
3. 最后：基于重排后的上下文生成答案

这一课的重点不是追求最高分，而是让你真正看懂：

- 为什么第一阶段和第二阶段要分开
- 为什么 reranking 往往比单纯 top-k 检索更稳
- 在真实数据上，reranking 到底改善了什么

In [ ]:
# 先清理 GPU 缓存。
# notebook 本身尽量不直接占用 GPU，主要通过 LM Studio 的本地服务调用模型。
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 连接 LM Studio

这一课会自动发现本地 OpenAI 兼容接口里的模型。
这里我会优先选择：

- `qwen3.5-35b-a3b` 作为 chat / reranker 模型
- 当前可用的 embedding 模型作为向量模型

In [ ]:
import json
import re
from collections import Counter
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

BASE_URL = 'http://10.0.0.63:1234/v1'
API_KEY = 'lm-studio'

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
model_ids = [m.id for m in client.models.list().data]
print('available models:', model_ids)

preferred_chat_models = ['qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred_chat_models if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower() or 'embedding' in m.lower()), None)

print('chat_model      =', chat_model)
print('embedding_model =', embedding_model)

if chat_model is None:
    raise RuntimeError('没有找到可用的 chat model。')
if embedding_model is None:
    raise RuntimeError('没有找到可用的 embedding model。')

## 2. 加载真实数据集

这里我们继续使用 Hugging Face 的 `squad`。
为了让 reranking 的效果更容易观察，我们还是先用一个小规模子集。

In [ ]:
raw_ds = load_dataset('squad', split='validation[:140]')
print(raw_ds)
print(raw_ds[0]['question'])
print(raw_ds[0]['answers']['text'][0])

In [ ]:
context_to_doc_id = {}
documents = []
eval_rows = []

for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        doc_id = len(documents)
        context_to_doc_id[context] = doc_id
        documents.append({'doc_id': doc_id, 'text': context, 'title': item['title']})
    else:
        doc_id = context_to_doc_id[context]

    if item['answers']['text']:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': item['answers']['text'][0].strip(),
            'title': item['title'],
        })

print('num_documents =', len(documents))
print('num_eval_rows =', len(eval_rows))

## 3. 生成文档向量

这一格和上一课类似，但现在它是第一阶段召回的基础。

In [ ]:
def get_embeddings(texts: List[str], batch_size: int = 16) -> np.ndarray:
    all_vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        batch_vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
        all_vectors.extend(batch_vectors)
        print(f'embedded {min(start + batch_size, len(texts))}/{len(texts)}')
    return np.vstack(all_vectors)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms

In [ ]:
doc_texts = [doc['text'] for doc in documents]
doc_embeddings = get_embeddings(doc_texts, batch_size=16)
doc_embeddings = l2_normalize(doc_embeddings)
print('doc_embeddings shape =', doc_embeddings.shape)

## 4. 第一阶段：embedding 检索

这一阶段的目标是“不要漏掉”，也就是先把可能相关的候选尽量召回。

In [ ]:
def first_stage_retrieve(question: str, top_k: int = 5):
    q_emb = get_embeddings([question], batch_size=1)
    q_emb = l2_normalize(q_emb)[0]
    scores = doc_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            'doc_id': int(documents[idx]['doc_id']),
            'title': documents[idx]['title'],
            'score': float(scores[idx]),
            'text': documents[idx]['text'],
        })
    return results

## 5. 第二阶段：LLM reranking

这里我们让本地大模型直接看：
- 问题是什么
- 候选文档有哪些

然后让它给出一个新的排序。

这就是教学版的 LLM reranker。真实系统里也会有专门的 cross-encoder / reranker 模型，但你现在先把这个流程吃透就很值。

In [ ]:
def shorten(text: str, max_chars: int = 450) -> str:
    text = text.replace('\n', ' ').strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + ' ...'


def rerank_with_llm(question: str, candidates, top_n: int = 3):
    docs_block = []
    for i, cand in enumerate(candidates, start=1):
        docs_block.append(
            f'DOC_{i} | doc_id={cand["doc_id"]} | title={cand["title"]}\n{shorten(cand["text"])}'
        )
    docs_block = '\n\n'.join(docs_block)

    system_prompt = (
        'You are a retrieval reranker. '
        'Given a question and candidate documents, rank the documents by how useful they are for answering the question. '
        'Return strict JSON only.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Candidates:\n{docs_block}\n\n'
        'Return a JSON object with one key named ranked_doc_ids. '
        'The value must be a list of doc_id integers ordered from best to worst.'
    )

    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content.strip()
    match = re.search(r'\{.*\}', raw, flags=re.S)
    if not match:
        raise ValueError(f'没有从 reranker 输出里找到 JSON: {raw}')
    parsed = json.loads(match.group(0))
    ranked_doc_ids = parsed['ranked_doc_ids']

    doc_map = {cand['doc_id']: cand for cand in candidates}
    reranked = [doc_map[doc_id] for doc_id in ranked_doc_ids if doc_id in doc_map]

    # 如果模型漏掉了某些候选，就把它们按原顺序补到后面。
    seen = {row['doc_id'] for row in reranked}
    for cand in candidates:
        if cand['doc_id'] not in seen:
            reranked.append(cand)

    return reranked[:top_n], raw

## 6. 生成答案

这一步和上一课类似，只是现在默认使用重排后的前几个文档来回答。

In [ ]:
def build_context_block(retrieved_docs):
    parts = []
    for i, doc in enumerate(retrieved_docs, start=1):
        parts.append(f'[Document {i}] title: {doc["title"]}\n{doc["text"]}')
    return '\n\n'.join(parts)


def answer_with_llm(question: str, retrieved_docs):
    system_prompt = (
        'You are a careful question-answering assistant. '
        'Answer only from the provided context. '
        'If the answer is not supported by the context, reply with NOT_FOUND. '
        'Keep the answer short.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{build_context_block(retrieved_docs)}\n\n'
        'Return only the answer.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

## 7. 先看一条完整流程

这一步你会同时看到：
- 第一阶段召回
- 第二阶段重排
- 最终答案

In [ ]:
sample = eval_rows[0]
first_stage = first_stage_retrieve(sample['question'], top_k=5)
second_stage, rerank_raw = rerank_with_llm(sample['question'], first_stage, top_n=3)
pred_answer = answer_with_llm(sample['question'], second_stage)

print('question:')
print(sample['question'])
print('\ngold_doc_id:', sample['gold_doc_id'])
print('reference_answer:', sample['reference_answer'])

print('\nfirst-stage top-5:')
for row in first_stage:
    print(f"doc_id={row['doc_id']} | score={row['score']:.4f} | title={row['title']}")

print('\nreranker raw json:')
print(rerank_raw)

print('\nsecond-stage top-3:')
for row in second_stage:
    print(f"doc_id={row['doc_id']} | title={row['title']}")

print('\npred_answer:')
print(pred_answer)

## 8. 评估函数

这节课我们重点比较两件事：

- `FirstStageHit@1` vs `SecondStageHit@1`
- 重排前后生成答案的变化

所以评估会保留：
- 检索命中
- `Exact Match`
- `Token F1`

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    return token


def tokenize(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(tokenize(text))


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 9. 小规模批量评估

因为这里每条都要多一次 rerank 调用，所以先只跑前 4 条。
你后面熟悉后可以自己慢慢往上加。

In [ ]:
eval_subset = eval_rows[:4]
results = []

for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')

    first_stage = first_stage_retrieve(row['question'], top_k=5)
    second_stage, rerank_raw = rerank_with_llm(row['question'], first_stage, top_n=3)

    answer_first = answer_with_llm(row['question'], first_stage[:3])
    answer_second = answer_with_llm(row['question'], second_stage)

    first_hit1 = 1.0 if first_stage[0]['doc_id'] == row['gold_doc_id'] else 0.0
    second_hit1 = 1.0 if second_stage[0]['doc_id'] == row['gold_doc_id'] else 0.0

    results.append({
        'question': row['question'],
        'gold_doc_id': row['gold_doc_id'],
        'first_stage_ids': [x['doc_id'] for x in first_stage],
        'second_stage_ids': [x['doc_id'] for x in second_stage],
        'first_hit@1': first_hit1,
        'second_hit@1': second_hit1,
        'reference_answer': row['reference_answer'],
        'answer_first': answer_first,
        'answer_second': answer_second,
        'em_first': exact_match(answer_first, row['reference_answer']),
        'em_second': exact_match(answer_second, row['reference_answer']),
        'f1_first': token_f1(answer_first, row['reference_answer']),
        'f1_second': token_f1(answer_second, row['reference_answer']),
    })

summary = {
    'FirstStageHit@1': float(np.mean([x['first_hit@1'] for x in results])),
    'SecondStageHit@1': float(np.mean([x['second_hit@1'] for x in results])),
    'FirstStageEM': float(np.mean([x['em_first'] for x in results])),
    'SecondStageEM': float(np.mean([x['em_second'] for x in results])),
    'FirstStageF1': float(np.mean([x['f1_first'] for x in results])),
    'SecondStageF1': float(np.mean([x['f1_second'] for x in results])),
}

print('\nsummary =', summary)

## 10. 看逐条结果

这是这节课最值得细看的地方。
你会看到 reranker 到底有没有把“更该排前面”的文档往前推。

In [ ]:
for row in results:
    print('question:', row['question'])
    print('gold_doc_id:', row['gold_doc_id'])
    print('first_stage_ids :', row['first_stage_ids'])
    print('second_stage_ids:', row['second_stage_ids'])
    print('first_hit@1 / second_hit@1:', row['first_hit@1'], '/', row['second_hit@1'])
    print('reference_answer:', row['reference_answer'])
    print('answer_first :', row['answer_first'])
    print('answer_second:', row['answer_second'])
    print('EM first/second:', row['em_first'], '/', row['em_second'])
    print('F1 first/second:', round(row['f1_first'], 3), '/', round(row['f1_second'], 3))
    print('-' * 100)

## 11. 这节课的工程直觉

你现在应该能看出两阶段结构的味道了：

- 第一阶段更像“广撒网”
- 第二阶段更像“精挑细选”

这也是为什么真实 RAG 里，很多团队都会把系统拆成：
- recall stage
- rerank stage
- answer stage

因为这三层的优化目标本来就不完全一样。

## 12. 本课小结

这节课你要带走的核心是：

1. 第一阶段 embedding 检索更偏召回。
2. 第二阶段 reranking 更偏排序质量。
3. 用本地大模型也可以直接做一个教学版 reranker。
4. 真实 RAG 往往不是单次检索就结束，而是多阶段逐步变精。

下一课最自然的衔接就是：
**真实版 Judge / LLM-as-a-Judge 评估**，也就是让本地模型帮你评估答案质量。